# Day 9 — File Caching

> ⚠️ **Why this matters.** Every API call costs time (network round-trip) and rate-limit budget. Caching the responses to disk means: faster lookups, works offline, kinder to the API. Real apps cache everything they can.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/09-caching.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min mini-project + 45 min quiz.

By the end:

- [ ] You can design a simple file-based cache
- [ ] You know when to invalidate (TTL) and when not to
- [ ] You've added a cache layer between english-helper and the API
- [ ] Your tool works offline for words you've already looked up
- [ ] You understand `functools.lru_cache` for in-memory caching

## The mental model

**Cache = remember an answer so you don't compute/fetch it again.**

```mermaid
graph LR
    User[lookup 'thorough'] --> Cache{In cache?}
    Cache -->|yes| Hit[Return cached]
    Cache -->|no| API[Fetch from API]
    API --> Store[Store in cache]
    Store --> Hit
```

**Three layers of caching you'll use:**

1. **In-memory** (`functools.lru_cache`): fast, lost on restart, limited capacity.
2. **Local file** (today): persists, works offline, manual invalidation.
3. **Distributed** (Redis, Memcached): for many machines — Phase 6+.

Today: layer 2. Built on what you learned Day 5 (JSON) + Day 7 (requests).

## 1. The simplest cache — `dict + json file`

In [ ]:
import json
from pathlib import Path

CACHE_FILE = Path.home() / '.english-helper' / 'api_cache.json'

def load_cache() -> dict[str, dict]:
    if not CACHE_FILE.exists():
        return {}
    return json.loads(CACHE_FILE.read_text())

def save_cache(cache: dict[str, dict]) -> None:
    CACHE_FILE.parent.mkdir(exist_ok=True)
    CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=2))

def cached_fetch(word: str, fetch_fn) -> dict | None:
    cache = load_cache()
    if word in cache:
        print(f'cache hit: {word}')
        return cache[word]
    print(f'cache miss: {word} — fetching')
    result = fetch_fn(word)
    if result is not None:
        cache[word] = result
        save_cache(cache)
    return result

This is functional. Pass any `fetch_fn` (like your `fetch_word`); calls go through cache transparently.

## 2. Cache invalidation — TTL

In [ ]:
import time
from pathlib import Path
import json

CACHE_FILE = Path('/tmp/cache_ttl.json')
TTL_SECONDS = 60 * 60 * 24 * 7   # one week

def is_fresh(entry: dict) -> bool:
    return time.time() - entry['stored_at'] < TTL_SECONDS

def cached_with_ttl(word: str, fetch_fn) -> dict | None:
    cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
    if word in cache and is_fresh(cache[word]):
        return cache[word]['data']
    data = fetch_fn(word)
    if data is not None:
        cache[word] = {'data': data, 'stored_at': time.time()}
        CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=2))
    return data

**For english-helper, you probably don't need TTL.** A dictionary definition isn't going to change. But you'd want TTL for: stock prices (seconds), weather (10 min), news (1 hr), user profiles (1 day).

> 💡 **In the wild:** Phil Karlton famously said "There are only two hard things in Computer Science: cache invalidation and naming things." When in doubt, **never expire**, or **let user manually clear**.

## 3. In-memory cache with `lru_cache`

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=100)
def expensive_lookup(word: str) -> str:
    # Imagine this is slow
    print(f'computing for {word}')
    return word.upper()

expensive_lookup('hello')  # prints 'computing for hello'
expensive_lookup('hello')  # cache hit, no print
expensive_lookup('world')  # prints 'computing for world'

**Use `lru_cache` when:**

- The function is pure (same inputs → same output, no side effects).
- The result fits in memory.
- You're calling it many times with the same args within a single run.

**Don't use `lru_cache` for:**

- Network calls in long-running programs (cache grows forever).
- Anything with side effects (DB writes, prints that matter).
- Functions with unhashable args (lists, dicts) — won't work.

## End-of-day mini-project — add caching to `api.py`

> 🎯 **Today's piece:** put a transparent disk cache between english-helper and the Free Dictionary API.

### What you're building

Add a new module `src/english_helper/cache.py`:

```python
def load_cache() -> dict[str, dict]: ...
def save_cache(cache: dict[str, dict]) -> None: ...
def get_or_fetch(word: str, fetch_fn) -> dict | None: ...
def clear_cache() -> int: ...   # returns number of entries removed
```

Then update `api.py`:

- `fetch_word_cached(word)` — uses the cache, calls `fetch_word` only on miss.
- Hook it into the `cli.py` `add` command so users get the cached version transparently.

Cache location: `~/.english-helper/api_cache.json`.

### Verify

```bash
$ uv run english-helper add ubiquitous
cache miss — fetching ubiquitous
Added.

$ uv run english-helper add ubiquitous
(Already in your vocabulary.)

$ rm -rf ~/.english-helper && uv run english-helper add ubiquitous
cache miss — fetching ubiquitous   # again, because cache was deleted
Added.
```

## Connect to the project

> 🎯 **Connects to the project:** Tomorrow (Day 10) you wire all of Week 2 together: real API + cache + your existing vocab_app CLI become one tool. By Friday your english-helper does the same thing dictionary apps do — but you built it.

**Quiz:** [09-caching-quiz.ipynb](09-caching-quiz.ipynb)